# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_mask = cat['product_name'].str.lower().str.contains('pop', na=False)
pop_tarts = cat[pop_mask]

print(f'Count of Pop-Tart rows: {len(pop_tarts)}')
display(pop_tarts)


Count of Pop-Tart rows: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
strawberry_mask = pop_tarts['product_name'].str.lower().str.contains('strawberry', na=False)

strawberry_pts = pop_tarts[strawberry_mask]
other_pts = pop_tarts[~strawberry_mask]

print('Strawberry Pop-Tart Variants (To Consolidate):')
for _, row in strawberry_pts.iterrows():
    print(f'{row["sku"]}: {row["product_name"]}')
print('\nOther Pop-Tart Variants (To Leave Alone):')
if len(other_pts) > 0:
    for _, row in other_pts.iterrows():
         print(f'{row["sku"]}: {row["product_name"]}')
else:
    print('None')

Strawberry Pop-Tart Variants (To Consolidate):
PT-12: pop-tarts strawberry
SKU#459812: Strawberry Pop-Tart
POPTART-STRAW: Pop-Tarts Strawberry

Other Pop-Tart Variants (To Leave Alone):
None


**Pop-Tart rows that are a different product:** _..._ Brown Sugar Cinnamon and Blueberry

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
from textwrap import fill
# TODO
canonical_sku = strawberry_pts['sku'].iloc[0]
sku_map = {sku: canonical_sku for sku in strawberry_pts['sku']}
print("SKU mapping dictionary:", sku_map)
cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])
display(cat.head(10))

SKU mapping dictionary: {'PT-12': 'PT-12', 'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12'}


,sku,product_name,category,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,PT-12
1,SKU#459812,Strawberry Pop-Tart,Breakfast,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print("Before Normalization")
print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = cat['category'].astype(str).str.strip().str.title()

print("\n After Strip & Title-Case")
print(cat['category_clean'].value_counts(dropna=False))

category_synonym_map = {
    'Breakfast Foods': 'Breakfast',
}
cat['category_final'] = cat['category_clean'].replace(category_synonym_map)

Before Normalization
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

 After Strip & Title-Case
category_clean
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


Breakfast and Breakfast Foods stay separate because teh string cleaning only fixes capitalization and punctuation.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
clean_names = (
    cat['product_name']
    .astype(str)
    .str.lower()
    .str.replace(r'[^a-z0-9\s]', ' ', regex=True)
)

cat['name_signature'] = clean_names.apply(
    lambda x: " ".join(sorted([w.rstrip('s') for w in x.split()]))
)

fragmented_groups = (
    cat.groupby('name_signature')
    .agg(
        sku_count=('sku', 'nunique'),
        skus=('sku', list),
        product_names=('product_name', list),
        categories=('category', list)
    )
    .query('sku_count > 1')
    .sort_values('sku_count', ascending=False)
)

print(f"Found {len(fragmented_groups)} fragmented product groups:")
display(fragmented_groups)

Found 1 fragmented product groups:


,sku_count,skus,product_names,categories
name_signature,,,,
pop strawberry tart,3,"[PT-12, SKU#459812, POPTART-STRAW]","[pop-tarts strawberry, Strawberry Pop-Tart, Po...","[Breakfast Foods, Breakfast, Breakfast]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
initial_row_count = len(cat)
assert cat['canonical_sku'].isna().sum() == 0, "ERROR: Null values found in canonical_sku!"
assert len(cat) == initial_row_count, "ERROR: Row count changed during mapping!"
print(f"Original unique SKUs:  {cat['sku'].nunique()}")
print(f"Canonical unique SKUs: {cat['canonical_sku'].nunique()}")
print("All safety assertions passed! Zero nulls introduced and row count preserved.")

Original unique SKUs:  3
Canonical unique SKUs: 1
All safety assertions passed! Zero nulls introduced and row count preserved.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

_your answer here_
a) Two produce families that appear fragmented are the soda/Coca-Cola and Milk/Dairy. The split in Soda/Coca-Cola is probably due to the hyphen, capitalization, maybe a flipped word. I would make sure everything was lowercase, remove all punctuation, and sort the words alphabetically to make sure the words were in correct order. In the case of dairy/milk they are probably split because of singluar vs plural, extra spaces, or inconsistant categories. I would group by lowercasing all letters, remove punctuation, and remove all plurals.

b) Data cleaning is when two items are the exact same, but just called different things like Strawberry Poptarts and Strawberry Pop-tarts. It becomes a bysiness decision when the items are actually different like Strawberry Pop-tarts - Frosted and Strawberry Pop-tarts - Unfrosted. In this example Strawberry pop-tarts don't list a ize of whether it is frosted so I can't tell if the listed products are teh exact same products. Before combining them I would ask the manager if they knew the SKUs or another defining characteristic.